# EX8 — Metoda Jakobiego jako przykład metody iteracyjnej

Rozważamy układ równań liniowych $A \cdot x = b$ i przedstawiamy **metodę Jakobiego** — jedną z podstawowych metod iteracyjnych solveowania układów liniowych.

Metoda polega na rozkładzie macierzy współczynników $A$ na trzy składowe:
$$A = D + L + U$$
gdzie:
- $D$ — macierz diagonalna (elemeny diagonalne $A$),
- $L$ — dolny trójkątny (zero na diagonalnej),
- $U$ — górny trójkątny (zero na diagonalnej).

Na tym rozkładzie opiera się schemat iteracyjny metody Jakobiego, który przedstawiamy w dwóch wariantach: pierwszy z jawnym $D^{-1}$, drugi — numerycznie stabilny.

**Cel:** Porównać dwa podejścia do implementacji metody Jakobiego i zilustrować, dlaczego w praktyce numerycznej unika się jawnego obliczania odwrotności macierzy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[15, -2, 3, 0, 6],
              [-3, 19, 1, -2, 7.4],
              [2, -1, -17, 1, 6.7],
              [4, 3, -5, 17, 9],
              [2, 3.5, 6.1, -4, -18.1]])
b = np.array([-1, 2, 3, 0.5, 13.1])

D = np.diag(np.diag(A))
L = np.tril(A, k=-1)
U = np.triu(A, k=1)

M = -np.linalg.inv(D) @ (L + U)
w = np.linalg.inv(D) @ b

x0 = np.random.rand(len(b))
delta0 = 1e-8
i_max = 50
x_old = x0.copy()
err = []
n_iter = [0]
i = 1
error = 5 * delta0

while (error >= delta0) and (i < i_max):
    x_new = M @ x_old + w
    error = np.linalg.norm(x_new - x_old)
    x_old = x_new
    err.append(error)
    n_iter.append(i)
    i += 1

plt.figure(figsize=(8, 5))
plt.semilogy(n_iter, [5*delta0] + err, 'b-o', linewidth=1.5, markersize=4)
plt.xlabel('# iteracji')
plt.ylabel('Estymata błędu')
plt.title('Metoda Jakobiego (naive) — estymata błędu')
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f'Liczba iteracji: {len(n_iter)}')
print(f'Rozwiązanie: {x_old.round(6)}')

## Wzór ogólny metody Jakobiego

Z zapisu $A = D + L + U$ równanie $A \cdot x = b$ przyjmuje postać:

$$D \cdot x + (L + U) \cdot x = b$$

Przekształcamy, izolując wyrażenie z $D$:

$$D \cdot x = b - (L + U) \cdot x$$

Mnożąc od lewej strony przez $D^{-1}$ otrzymujemy schemat iteracyjny:

$$x_{k+1} = -D^{-1}(L + U) \cdot x_k + D^{-1} b$$

W skrócie, oznaczając $M = -D^{-1}(L+U)$ oraz $w = D^{-1}b$:

$$x_{k+1} = M \cdot x_k + w \qquad \text{(forma macierzowa)}$$

W kodzie z komórki 2 macierz $M$ i wektor $w$ są obliczane jawnie za pomocą `np.linalg.inv(D)`. Jednak jak pokażemy niżej, takie podejście jest numerycznie nieoptymalne.

## Uwaga: Stabilność numeryczna — unikanie jawnego odwrotnego

Jawne obliczanie $D^{-1}$ jest powszechnie uznawane za **złą praktykę numeryczną** z kilku powodów:

1. **Błędy zaokrągleń:** Obliczanie odwrotności wprowadza dodatkowe błędy zaokrągleń, szczególnie gdy wyznacznik jest mały.
2. **Koszt obliczeniowy:** Obliczenie odwrotności kosztuje $O(n^3)$, podczas gdy rozwiązanie równania kosztuje $O(n^2)$ (dla macierzy trójkątnej / diagonalnej nawet mniej).

Zamiast:

$$x_{k+1} = -D^{-1}(L+U) \cdot x_k + D^{-1} b$$

Zalecamy postać:

$$D \cdot x_{k+1} = b - (L+U) \cdot x_k$$

Którą rozwiązujemy iteracyjnie za pomocą `np.linalg.solve(D, b - (L+U) @ x_k)`. W kolejnej komórce pokazujemy, jak wygląda implementacja z tą korekcją.

In [ ]:
A = np.array([[15, -2, 3, 0, 6],
              [-3, 19, 1, -2, 7.4],
              [2, -1, -17, 1, 6.7],
              [4, 3, -5, 17, 9],
              [2, 3.5, 6.1, -4, -18.1]])
b = np.array([-1, 2, 3, 0.5, 13.1])

D = np.diag(np.diag(A))
L = np.tril(A, k=-1)
U = np.triu(A, k=1)

x0 = np.random.rand(len(b))
delta0 = 1e-8
i_max = 50
x_old = x0.copy()
err = []
n_iter = [0]
i = 1
error = 5 * delta0

while (error >= delta0) and (i < i_max):
    x_new = np.linalg.solve(D, b - (L + U) @ x_old)
    error = np.linalg.norm(x_new - x_old)
    x_old = x_new
    err.append(error)
    n_iter.append(i)
    i += 1

plt.figure(figsize=(8, 5))
plt.semilogy(n_iter, [5*delta0] + err, 'r-s', linewidth=1.5, markersize=4)
plt.xlabel('# iteracji')
plt.ylabel('Estymata błędu')
plt.title('Metoda Jakobiego (stabilna) — estymata błędu')
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f'Liczba iteracji: {len(n_iter)}')
print(f'Rozwiązanie: {x_old.round(6)}')

## Wykorzystywane funkcje

| Funkcja | Opis |
|---------|------|
| `np.diag(A)` | Zwraca diagonalę z macierzy / tworzy macierz diagonalną |
| `np.tril(A)` | Dolny trójkąt macierzy |
| `np.triu(A)` | Górny trójkąt macierzy |
| `np.linalg.inv(D)` | Odwraca macierz $D$ |
| `np.linalg.solve(D, rhs)` | Rozwiązuje układ $D \cdot x = rhs$ bez jawnej odwrotności |
| `np.linalg.norm(x)` | Norma euklidesowa (L2) wektora |

**Podsumowanie:** Obie implementacje dają ten sam wynik liczbowy, ale wersja z `np.linalg.solve` jest preferowana ze względów stabilności numerycznej. Zasada _"nie obliczaj odwrotności macierzy, rozwiąż układ"_ jest powszechnie uznawanym standardem w obliczeniach naukowych.